# Notebook 2: Data Preprocessing and Feature Engineering

**Project**: E-Commerce Fraud Detection Analysis  
**Subject**: Fraud Detection Analysis (MBA Data Science)  
**Dataset**: IEEE-CIS Fraud Detection Dataset  

---

## 1. Overview & Objectives
This notebook prepares the raw transaction dataset for machine learning by applying temporal train/validation splitting and engineering domain-justified features.


In [1]:
# Imports, Setup & Preprocessing Functions
import os
import pandas as pd
import numpy as np

data_dir = 'data/raw'
proc_dir = 'data/processed'
os.makedirs(proc_dir, exist_ok=True)

def reduce_mem_usage(df, verbose=False):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.api.types.is_integer_dtype(df[col]):
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min >= np.iinfo(np.int64).min and c_max <= np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    if verbose:
        end_mem = df.memory_usage().sum() / 1024**2
        print(f'Memory usage decreased to {end_mem:.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

def load_and_merge_data(data_dir, nrows=None):
    trans_path = os.path.join(data_dir, 'train_transaction.csv')
    id_path = os.path.join(data_dir, 'train_identity.csv')
    print('Loading train_transaction.csv...', flush=True)
    trans_df = pd.read_csv(trans_path, nrows=nrows)
    print('Loading train_identity.csv...', flush=True)
    id_df = pd.read_csv(id_path, nrows=nrows)
    print(f'Left joining transaction ({len(trans_df):,} rows) and identity ({len(id_df):,} rows)...', flush=True)
    merged_df = pd.merge(trans_df, id_df, on='TransactionID', how='left')
    merged_df = reduce_mem_usage(merged_df, verbose=True)
    print(f'Merged dataset shape: {merged_df.shape}')
    return merged_df

def split_data_temporal(df, val_size=0.2, time_col='TransactionDT'):
    df_sorted = df.sort_values(by=time_col).reset_index(drop=True)
    split_idx = int(len(df_sorted) * (1 - val_size))
    train_df = df_sorted.iloc[:split_idx].copy()
    val_df = df_sorted.iloc[split_idx:].copy()
    print(f'Temporal Split: Train={len(train_df):,} rows, Validation={len(val_df):,} rows', flush=True)
    return train_df, val_df

def engineer_features(df):
    df = df.copy()
    df['transaction_hour'] = ((df['TransactionDT'] // 3600) % 24).astype(np.int8)
    df['log_TransactionAmt'] = np.log1p(df['TransactionAmt']).astype(np.float32)
    identity_cols = [c for c in df.columns if c.startswith('id_') or c in ['DeviceType', 'DeviceInfo']]
    if identity_cols:
        df['missing_identity_flag'] = df[identity_cols].isnull().all(axis=1).astype(np.int8)
    else:
        df['missing_identity_flag'] = 1
    df['missing_count_row'] = df.isnull().sum(axis=1).astype(np.int16)
    if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
        df['email_match_flag'] = (
            (df['P_emaildomain'] == df['R_emaildomain']) & 
            df['P_emaildomain'].notnull() & 
            df['R_emaildomain'].notnull()
        ).astype(np.int8)
    else:
        df['email_match_flag'] = 0
    return df

raw_df = load_and_merge_data(data_dir)
train_df, val_df = split_data_temporal(raw_df)
train_df = engineer_features(train_df)
val_df = engineer_features(val_df)

target_col = 'isFraud'
y_train = train_df[target_col].values
y_val = val_df[target_col].values
X_train = train_df.drop(columns=[target_col])
X_val = val_df.drop(columns=[target_col])

X_train.to_parquet(os.path.join(proc_dir, 'X_train.parquet'))
X_val.to_parquet(os.path.join(proc_dir, 'X_val.parquet'))
np.save(os.path.join(proc_dir, 'y_train.npy'), y_train)
np.save(os.path.join(proc_dir, 'y_val.npy'), y_val)
print('Saved processed parquet and npy arrays to data/processed/')


Loading train_transaction.csv...
Loading train_identity.csv...
Left joining transaction (590,540 rows) and identity (144,233 rows)...
Memory usage decreased to 1073.53 MB (45.9% reduction)
Merged dataset shape: (590540, 434)
Temporal Split: Train=472,432 rows, Validation=118,108 rows
Saved processed parquet and npy arrays to data/processed/
